# 04 系统蓝图：多工况锂电池 RUL 深度学习系统

目标：把研究流程从 Notebook 验证升级为**可复现实验系统**。

本 Notebook 对应的模块化代码在 `src/rul_system/`：
- `data.py`：数据加载、工况编码、窗口化
- `models.py`：条件感知 Transformer
- `train.py`：训练、预测与指标

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = r"C:\\Users\\PLUTO\\Desktop\\battery-rul"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.rul_system.data import (
    load_battery_raw_parameters,
    default_condition_map,
    prepare_condition_aware_dataloaders,
)
from src.rul_system.models import ConditionAwareTransformer
from src.rul_system.train import train_model, predict, compute_regression_metrics

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1) 数据与实验协议

这里给一个可执行的起点：
- 训练：室温电池 + 部分低温电池
- 测试：低温未见电池 `B0053`（跨工况外推）

你后续可以替换为 LOBO/LOCO 更严格协议。

In [ ]:
DATA_DIR_GROUP1 = os.path.join(PROJECT_ROOT, "data\\processed\\1. BatteryAgingARC-FY08Q4")
DATA_DIR_GROUP6 = os.path.join(PROJECT_ROOT, "data\\processed\\6. BatteryAgingARC_53_54_55_56")

room_ids = ["B0005", "B0006", "B0007", "B0018"]
cold_ids = ["B0053", "B0054", "B0055", "B0056"]

room_data = load_battery_raw_parameters(DATA_DIR_GROUP1, room_ids)
cold_data = load_battery_raw_parameters(DATA_DIR_GROUP6, cold_ids)
all_data = {**room_data, **cold_data}

condition_map = default_condition_map()

train_ids = ["B0005", "B0006", "B0007", "B0018", "B0054", "B0055", "B0056"]
test_id = "B0053"

print("Train IDs:", train_ids)
print("Test ID:", test_id)

## 2) 构建 DataLoader（训练集拟合缩放器，测试集仅变换）

In [ ]:
SEQ_LENGTH = 10
BATCH_SIZE = 32

train_loader, test_loader, scaler_y, test_actual_capacity = prepare_condition_aware_dataloaders(
    battery_dict=all_data,
    train_ids=train_ids,
    test_id=test_id,
    condition_map=condition_map,
    seq_length=SEQ_LENGTH,
    batch_size=BATCH_SIZE,
)

print("Train batches:", len(train_loader))
print("Test samples:", len(test_loader))

## 3) 模型训练（条件感知 Transformer）

In [ ]:
model = ConditionAwareTransformer(
    input_dim=5,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.1,
    max_seq_len=128,
)

history = train_model(
    model=model,
    train_loader=train_loader,
    epochs=40,
    lr=1e-3,
    device=device,
)

print("Final train loss:", history[-1])

## 4) 测试评估与可视化

In [ ]:
preds_real = predict(model, test_loader, scaler_y=scaler_y, device=device)
y_true = test_actual_capacity[SEQ_LENGTH:]
metrics = compute_regression_metrics(y_true, preds_real)

print("==== Test Metrics (" + test_id + ") ====")
for k, v in metrics.items():
    suffix = "%" if k == "MAPE" else "Ah"
    print(f"{k}: {v:.4f} {suffix}")

plt.figure(figsize=(10, 5))
plt.plot(range(len(test_actual_capacity)), test_actual_capacity, label="Ground Truth", color="blue")
plt.plot(range(SEQ_LENGTH, len(test_actual_capacity)), preds_real, label="Prediction", color="red", linestyle="--")
plt.axhline(y=1.4, color="black", linestyle=":", label="EOL (1.4 Ah)")
plt.title(f"Condition-Aware Transformer on {test_id}")
plt.xlabel("Cycle Index")
plt.ylabel("Capacity (Ah)")
plt.legend()
plt.grid(True)
plt.show()

## 5) 研究实践说明（为什么这是经典做法）

是的，这是非常典型的科研工程化路径：
1. Notebook 做假设验证与结果展示；
2. 把稳定逻辑沉淀成 `.py` 模块；
3. 每次实验只改配置与协议，不改核心代码；
4. 最终形成可复现实验流水线。